In [12]:
import h5py
import pandas as pd



filename = "../data/ccnews-nomic-768-normalized.hdf5"
with h5py.File(filename, "r") as f:
    f.visititems(lambda name, obj: print(name, type(obj).__name__))

avg_distances Dataset
distances Dataset
neighbors Dataset
test Dataset
train Dataset


In [13]:
import h5py

with h5py.File(filename, "r") as f:
    for name in f.keys():
        dataset = f[name]
        print(f"{name}: shape={dataset.shape}, dtype={dataset.dtype}")




avg_distances: shape=(1000,), dtype=float64
distances: shape=(1000, 100), dtype=float64
neighbors: shape=(1000, 100), dtype=int64
test: shape=(1000, 768), dtype=float32
train: shape=(495328, 768), dtype=float32


#### avg_distances data


In [14]:
with h5py.File(filename, "r") as f:
    avg_distances = f["avg_distances"][:]
    df = pd.DataFrame({
        "avg_distances": f["avg_distances"][:]
    })
df

,avg_distances
0,0.451584
1,0.499857
2,0.442227
3,0.476359
4,0.450745
...,...
995,0.465523
996,0.436732
997,0.471668
998,0.444338


#### neighbors data

In [15]:
with h5py.File(filename, "r") as f:
    neighbors = f["neighbors"][:]
neighbors

array([[331646, 460655, 118234, ..., 437574, 106239, 143833],
       [161490,   9045, 186076, ..., 279553, 308170, 431997],
       [166831, 386430,  43554, ..., 438309,  43454, 233082],
       ...,
       [427159, 325002, 366355, ..., 100105, 272313, 353171],
       [339577, 202547, 473597, ..., 218467, 157849, 456737],
       [283204, 338241, 424452, ..., 358471, 456716, 313805]],
      shape=(1000, 100))

#### distances data

In [16]:
with h5py.File(filename, "r") as f:
    distances = f["distances"][:]
distances

array([[0.14955997, 0.1539191 , 0.15749824, ..., 0.23119557, 0.23120278,
        0.23123461],
       [0.15324378, 0.15457773, 0.17762178, ..., 0.23270059, 0.23299837,
        0.2333324 ],
       [0.1826036 , 0.18334895, 0.1835497 , ..., 0.22431016, 0.22432709,
        0.224415  ],
       ...,
       [0.19622707, 0.20057344, 0.21564436, ..., 0.27498811, 0.27506095,
        0.27509642],
       [0.11687195, 0.12650657, 0.14442331, ..., 0.20604527, 0.2060551 ,
        0.20624042],
       [0.00360012, 0.06471097, 0.07577091, ..., 0.27093571, 0.27110434,
        0.27142084]], shape=(1000, 100))

#### test data

In [17]:
with h5py.File(filename, "r") as f:
    test = f["test"][:]
test

array([[-0.01059632,  0.00122034, -0.20069289, ..., -0.03269816,
        -0.04930089,  0.00900531],
       [-0.01504593,  0.06016366, -0.17010768, ..., -0.06284764,
        -0.00944398, -0.03868816],
       [ 0.04431966,  0.05338814, -0.20244338, ..., -0.03894779,
        -0.04877932, -0.04583669],
       ...,
       [-0.01991585,  0.11006858, -0.2110422 , ..., -0.03737114,
        -0.05588567, -0.00409174],
       [ 0.01942561,  0.07956424, -0.19632664, ..., -0.09493148,
        -0.04685321, -0.02495656],
       [ 0.07469226,  0.04301128, -0.19795196, ..., -0.07251247,
        -0.07564262, -0.04969097]], shape=(1000, 768), dtype=float32)

#### train data


In [18]:
with h5py.File(filename, "r") as f:
    train = f["train"][:]
train

array([[ 0.00118432,  0.04939296, -0.21632339, ..., -0.08226312,
        -0.02146901,  0.0117222 ],
       [ 0.03834809,  0.06859308, -0.20904763, ..., -0.07001527,
        -0.032758  , -0.05263406],
       [ 0.09853712,  0.01078159, -0.19446076, ..., -0.0535744 ,
        -0.03236951,  0.00206598],
       ...,
       [-0.05284675,  0.04655493, -0.1703702 , ..., -0.03749304,
        -0.01765583, -0.00363127],
       [-0.03565509,  0.04042041, -0.2126845 , ..., -0.07575109,
        -0.0445942 ,  0.00551131],
       [ 0.01178642,  0.05426823, -0.19343586, ..., -0.08533015,
        -0.04836286, -0.06108024]], shape=(495328, 768), dtype=float32)

In [19]:
import ecpfs

In [20]:
import numpy as np
def write_h5_embeddings(path, embeddings, dataset_name="embeddings"):
    with h5py.File(path, "w") as f:
        f.create_dataset(dataset_name, data=np.asarray(embeddings, dtype=np.float32))

write_h5_embeddings('embeddings.h5', train)

In [21]:
from ecpfs import Builder, Metric, EmbeddingDtype


builder = Builder(
    index_path="my_index.zarr",
    levels=3,
    metric=Metric.L2,
    is_normalized=False,
    # None stores each file's own dtype. A narrower EmbeddingDtype
    # (F16, UInt8, Int8) saves disk but loses precision, and logs a
    # warning. Every read widens back to f32, so it saves disk, not memory.
    embedding_dtype=None,
)



builder.select_representatives(
    embeddings_file="embeddings.h5",
    target_cluster_items=100,
    strategy="offset",
    fallback_batch_rows=100_000,
)

builder.build(
    embeddings_file="embeddings.h5",
    fallback_batch_rows=100_000,
)


In [22]:
from ecpfs import Index

import numpy as np
from ecpfs import Index

with Index("my_index.zarr") as index:
    query = np.random.rand(768).astype(np.float32)
    items, query_id = index.new_search(
        query=query,
        k=10,
        search_exp=4,
        max_increments=-1,
        exclude_vec=[],
    )
    # items: list[(score, item_id)], lower score is a better match.
    # See the search parameters page for k, search_exp and max_increments.

    # Pull further results for the same query without re-searching from the root:
    more_items = index.get_next_k_items(
        query_id=query_id,
        k=10,
        search_exp=4,
        max_increments=-1,
        exclude_vec=[],
    )
more_items

[(16.242420196533203, 85327),
 (16.242420196533203, 85327),
 (16.242647171020508, 200),
 (16.242647171020508, 200),
 (16.242647171020508, 200),
 (16.242647171020508, 200),
 (16.243139266967773, 250251),
 (16.243139266967773, 250251),
 (16.243139266967773, 250251),
 (16.243139266967773, 250251)]